# SignSage v2 — Kaggle training run

Runs `research/training/train_sign_to_text.py` (corrected post-352868924 config).
Read `research/training/READINESS.md` first. Requires a Kaggle GPU + `HF_TOKEN` secret.
Each step below is one cell; outputs are the run record. Resume after interruption with `--resume`.

In [ ]:
!pip install -q pose-format jiwer sacrebleu rouge-score sentencepiece huggingface_hub requests tqdm pandas

In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Private-repo clone: GITHUB_TOKEN via Kaggle Secrets/env, used ONLY for
# this clone through a one-shot HTTP Authorization header. The persisted
# remote stays clean (https://github.com/therandomuser03/signsage.git) and
# the token is never printed, never in history, never in .git/config.
import base64
import json as _json
import os
import subprocess
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/therandomuser03/signsage.git"
DEST = Path("/kaggle/working/signsage")
TRAIN_SCRIPT = "research/training/train_sign_to_text.py"

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
print("GITHUB_TOKEN available:", bool(token))
if not token:
    raise RuntimeError("GITHUB_TOKEN is missing: add it under the Kaggle notebook Add-ons > Secrets, then re-run this cell.")

# Preflight: confirm the token can actually read the private repo.
try:
    _req = urllib.request.Request(
        "https://api.github.com/repos/therandomuser03/signsage",
        headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(_req, timeout=20) as _r:
        _info = _json.loads(_r.read().decode())
except Exception as exc:
    raise RuntimeError("Cannot access private repo therandomuser03/signsage with GITHUB_TOKEN (invalid/expired token or missing repo permission).") from exc
print("private repo accessible:", _info.get("full_name"), "| private:", _info.get("private"))

if not (DEST / ".git").exists():
    _cred = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {_cred}",
                    "clone", REPO_URL, str(DEST)], check=True)
    del _cred
del token

# Post-conditions: clean remote URL, expected training file present.
_remote = subprocess.run(["git", "-C", str(DEST), "remote", "get-url", "origin"],
                         capture_output=True, text=True, check=True).stdout.strip()
assert _remote == REPO_URL, f"remote URL polluted: {_remote!r}"
assert (DEST / TRAIN_SCRIPT).is_file(), f"missing {TRAIN_SCRIPT} in cloned repo"
print("clone OK; remote clean; training script present")

OUT = "/kaggle/working/s2t_v2"
os.makedirs(OUT, exist_ok=True)
print("repo + out ready (run the provisioning cell next)")


In [ ]:
# Dataset provisioning: resolve iSign_v1.1.csv from (1) an existing working
# copy, (2) an attached Kaggle dataset (set ISIGN_DATASET_SLUG if attached
# under a custom slug), or (3) a Hugging Face download. Fails early with a
# clear error when none is available -- training never starts without data.
# Also guarantees HF_TOKEN is exported for pose streaming (--pose-source
# remote always needs it): subprocesses inherit os.environ, plain Python
# locals do not. The token itself is never printed.
import os
from pathlib import Path

def _resolve_csv():
    cands = [Path("/kaggle/working/isign/iSign_v1.1.csv")]
    slug = os.environ.get("ISIGN_DATASET_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "iSign_v1.1.csv")
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").glob("*/iSign_v1.1.csv"))
    for _p in cands:
        if _p.is_file():
            return _p
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = os.environ.get("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("HF_TOKEN Kaggle Secret (or env) is required to download iSign from Hugging Face.") from exc
    if not _tok:
        raise RuntimeError("HF_TOKEN is empty: attach the CSV as a Kaggle dataset or set the HF_TOKEN secret.")
    os.environ["HF_TOKEN"] = _tok
    del _tok
    from huggingface_hub import hf_hub_download
    _dest = Path("/kaggle/working/isign")
    _dest.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(repo_id="Exploration-Lab/iSign", filename="iSign_v1.1.csv",
                                repo_type="dataset", token=os.environ["HF_TOKEN"], local_dir=str(_dest)))

CSV_PATH = str(_resolve_csv())
print("iSign CSV:", CSV_PATH)
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient as _USC
        _t = _USC().get_secret("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
        del _t
    except Exception as exc:
        raise RuntimeError("HF_TOKEN is required for remote pose streaming (--pose-source remote): set the HF_TOKEN Kaggle Secret, then re-run.") from exc
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is empty: set the HF_TOKEN Kaggle Secret for remote pose streaming, then re-run.")
print("HF_TOKEN available for pose streaming:", True)


In [ ]:
# Preflight (re-run any time): prove a REAL subprocess inherits HF_TOKEN
# before any expensive command. Reports availability + length ONLY --
# the token value is never printed, written, or embedded anywhere.
import os
import subprocess
import sys

_tok = os.environ.get("HF_TOKEN")
print("HF_TOKEN available:", bool(_tok))
if not _tok:
    raise RuntimeError("HF_TOKEN missing in kernel env: re-run the provisioning cell above (it pulls the Kaggle Secret into os.environ), then re-run this cell.")
_probe = subprocess.run(
    [sys.executable, "-c",
     "import os; _t=os.environ.get('HF_TOKEN'); "
     "print('subprocess HF_TOKEN available:', bool(_t)); "
     "print('token length:', len(_t) if _t else 0)"],
    capture_output=True, text=True)
print(_probe.stdout.strip())
if _probe.returncode != 0 or "subprocess HF_TOKEN available: True" not in _probe.stdout:
    raise RuntimeError("subprocess cannot see HF_TOKEN: restart the kernel, run cells top-to-bottom (provisioning, then this cell), and retry.")
print("preflight OK: training subprocesses will inherit HF_TOKEN")


In [ ]:
# Pose-source preflight (cheap, re-run any time): proves the remote pose
# archive is reachable via HTTP Range reads AND that one REAL pose member
# decodes end-to-end -- before any expensive stage. Writes
# /kaggle/working/s2t_v2/pose_preflight.json and exits; no training starts.
# Requires: provisioning + HF_TOKEN preflight cells above.
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --preflight-only

In [ ]:
!cd /kaggle/working/signsage && python research/training/audit_dataset.py --csv {CSV_PATH} --out /kaggle/working/s2t_v2/dataset_audit.json

In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Overfit gate only: aborts here (no GPU wasted) if plumbing is broken.
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --stages tokenizer,overfit

In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Full run (graceful stop at --max-hours). RESUME ACROSS SESSIONS:
# /kaggle/working is ephemeral -- --resume alone does NOT survive a new
# session. After each session: Save Version (output) to keep $OUT, then in
# the next session attach that output as input and copy it back, e.g.:
#   !cp -r /kaggle/input/<prior-output-slug>/s2t_v2 /kaggle/working/s2t_v2
# then re-run this same command with --resume (checkpoints + tokenizer
# are reloaded from /kaggle/working/s2t_v2).
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --epochs 12 --max-hours 8 --resume


In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Evaluate + export (runs automatically at the end of train, or standalone):
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --stages evaluate,export
import json
rep = json.load(open("/kaggle/working/s2t_v2/evaluation_report.json"))
print("test:", rep["test_metrics"])
print("diversity:", rep["diversity"])
print("gate:", rep["gate"])

In [ ]:
# Verify the exported bundle loads through the REAL serving adapter (no notebook state).
import sys; sys.path.insert(0, "/kaggle/working/signsage/backend")
import numpy as np
from app.ml.adapters.sign_to_text_adapter import SignToTextAdapter
a = SignToTextAdapter.load("/kaggle/working/s2t_v2/signsage_deployment_bundle", device="cpu")
r = a.transcribe(np.random.default_rng(0).normal(size=(25, 543, 3)).astype(np.float32))
print("bundle loads + transcribes:", r.accepted, r.reason, repr(r.text[:80]))
print("NOTE: random-input output says nothing about quality; see evaluation_report.json.")